In [ ]:
###############################
# Import libraries
###############################
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import scipy
from tqdm import tqdm
import numpy as np
import gc
import random
import pickle
from glob import *
import numpy as np
from torch import tensor as tt
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import Dataset, DataLoader
from matplotlib.pyplot import *
from IPython.display import clear_output
def t2n(x): return x.squeeze().cpu().detach().numpy()
import seaborn as sns
sns.set(rc={"figure.figsize": (8, 5)})
sns.set_style("darkgrid")

###############################
# Import the dependencies
###############################
from ReasoningCombinatorials.dfs_approach.sudoku.board_enumeration import *

%cd /content/ReasoningCombinatorials/sudoku_tools
!python setup.py build_ext --inplace

import sys
sys.path.append('content/ReasoningCombinatorials/sudoku_tools')
from sudoku_tools import *

###############################
# Dataset Creation
###############################

#++++++++++++++++++++
# [vocabulary]
# 0:      'end' of sequence
# 1-99:   'level' of guess
# 100:    'start' of sequence
# 101:    'rules end'
# 102:    'dead end'
# 111-999: moves
#++++++++++++++++++++

class SudokuDataset(torch.utils.data.Dataset):

    def __len__(self):
        return 9999  # Some large number to allow sampling

    def __getitem__(self,idx):
        #==============
        max_len = 250  # maximum allowed length of generated sequence
        negatives = 0  # whether the transcript will include removed tokens during backtracking (default=0)
        levels = 1     # whether the transcript will output guess-levels (default=1)
        #==============

        # generate Sudoku boards and their solution paths until a <guess> is included in the path (speeds up training when seeing more often cases with guesses)
        has_guess = False
        while not has_guess:

            # generate an unsolved Sudoku board
            sudoku_id = int(np.random.randint(18383222420692992, dtype=np.int64)) +  int(np.random.randint(1*2*3*4*5*6*7*8*9, dtype=np.int64)) * 18383222420692992
            solution = board_generate( sudoku_id )
            solution_np = np.array([int(i) for i in solution], dtype=np.int32)
            s = generate_sudoku(solution_np)

            # get the solution path (dfs transcript)
            X, Y = sudoku_path(s, max_len, negatives, levels)
            X = X[0]   # X contains the dfs transcript; X.shape= (max_len,)
            Y = Y[0]   # Y contains the one-hot target vectors for each token in X; Y.shape=(max_len,2000)

            if (101 in X) and (0 in X): has_guess=True  # 101 : rules_end  (therefore a guess will follow)

        l = np.where(X==0)[0][0]  # length of generated path until the 'end' of sequence
        X[l+1:] = 1000            # after the 1st '0' which indicates 'end' of sequence, fill with '1000' (so that '0' is unique)

        return l,X,Y


# print an Example
dataset = SudokuDataset()
dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=True)

l,X,Y = next(iter(dataset))
print(l,X,Y)

In [ ]:
###############################
# Import the dependencies
###############################
from ReasoningCombinatorials.dfs_approach.sudoku.kaggle_dataset import download_and_load_kaggle_dataset
K = download_and_load_kaggle_dataset()
from ReasoningCombinatorials.dfs_approach.sudoku.transformer_model import *
from ReasoningCombinatorials.dfs_approach.sudoku.optimizer import *
from ReasoningCombinatorials.dfs_approach.sudoku.evaluation import *

###############################
# Training loop
###############################
logs = 'logs/'
os.makedirs(logs, exist_ok=True)

# Dataloaders
#===============
bs_train = 32
bs_test = 64
#===============
dataloader = torch.utils.data.DataLoader(dataset, batch_size=bs_train, shuffle=True)
dataloader_test = torch.utils.data.DataLoader(dataset, batch_size=bs_test, shuffle=True)

losses, cell_accs, cell_accs_0, cell_accs_102, cell_accs_101, cell_accs_lvl = [], [], [], [], [], []
board_accs_random, board_accs_kaggle_3m = [], []
best_board_acc_random, best_board_acc_kaggle_3m  = 0, 0

for i in tqdm(range(total_steps)):
    model.train()
    torch.cuda.empty_cache()

    l,X,Y = next(iter(dataloader))

    X,Y = X.to(device), Y.bool().to(device)
    Y = Y[:,:,:1001]

    #===================================
    opt.zero_grad()
    out = model(X)
    z = torch.log_softmax(out, dim=-1)

    loss = -z[Y].mean()  # Cross-Entropy loss

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    scheduler.step()
    #===================================
    if i % 100 == 0:
        clear_output(wait=True)
        losses.append(loss.item())
        recent_loss = np.mean(losses[-30:])
        current_lr = opt.param_groups[0]['lr']
        plot(losses), grid(True), title(f'loss: {recent_loss:2.2f} | lr: {current_lr:2.5f}'), show()

        cell_acc, cell_acc_0, cell_acc_102, cell_acc_101, cell_acc_lvl = eval_cell_acc()
        cell_accs.append(cell_acc)
        cell_accs_0.append(cell_acc_0)
        cell_accs_102.append(cell_acc_102)
        cell_accs_101.append(cell_acc_101)
        cell_accs_lvl.append(cell_acc_lvl)

        recent_cell_acc = np.mean(cell_accs[-30:])
        plot(cell_accs_101), plot(cell_accs_102), plot(cell_accs_lvl), plot(cell_accs), legend(['101','102','lvl','all']),
        grid(True), title(f'cell_acc: {recent_cell_acc:2.3f}'), show()

    if i % 2000 == 0:
        #================================================
        #              Board Accuracies
        #================================================
        if recent_cell_acc > 0.9:  # start computing Board Accuracy only if cell accuracy is high enough

            board_acc_random = eval_board_acc(dset='random')
            board_acc_kaggle_3m = eval_board_acc(dset='kaggle_3m')

            board_accs_random.append(board_acc_random)
            board_accs_kaggle_3m.append(board_acc_kaggle_3m)

            recent_board_acc_random = np.mean(board_accs_random[-30:])
            plot(board_accs_random),
            grid(True), title(f'board_accs_random: {recent_board_acc_random:2.3f}'), show()

            recent_board_acc_kaggle_3m = np.mean(board_accs_kaggle_3m[-30:])
            plot(board_accs_kaggle_3m),
            grid(True), title(f'board_accs_kaggle_3m: {recent_board_acc_kaggle_3m:2.3f}'), show()

            # save best model
            if (recent_board_acc_random > best_board_acc_random) and (len(board_accs_random) > 30):
                best_board_acc_random = recent_board_acc_random
                torch.save(model.state_dict(),logs+'best_model_random.pt')
                torch.save(opt.state_dict(),logs+'best_opt.pt')
                torch.save(scheduler.state_dict(),logs+'best_scheduler.pt')


    if i % 1000 == 0:
        torch.save(model.state_dict(),logs+'best_model_recent.pt')
        torch.save(opt.state_dict(),logs+'best_opt_recent.pt')
        torch.save(scheduler.state_dict(),logs+'best_scheduler_recent.pt')

    if i % 100000 == 0:
        torch.save(model.state_dict(),f'{logs}model_chk_{i}.pt')